# Mid-term hydro scheduling — 4-node model results

Plots for the in-house SDDP mid-term scheduler (`midterm_sddp4.jl`): **ES, PT, FR, EU**
— a hybrid system: **ES and PT use the SMS++/plan4res dataset** the market chain runs
on (`Data/smspp_in` capacities/costs/demand + EDF 37-climate-year profiles), while
**FR and EU use the real ~2024 system** (`Data/Generation.csv` etc. + EMPIRE 2015–2019
series, thermal derated by `[midterm4.availability]`).  78 weekly stages from July
2024, 2-hour blocks, 3 reservoir states (ES / FR / EU); each of the 37 Iberian climate
years is paired with a fixed EMPIRE year.  The merged capacity/cost table actually
used is written to `Data/midterm4_effective_inputs.csv`.

**Scenario-aware:** the loader below reads `config.toml [scenario] label` and appends
`_<label>` to every input path, so it plots whichever system `midterm_sddp4.jl` last
exported. `label = "2024"` reads the un-suffixed files; e.g. `label = "NECP2035"` reads
`..._sddp4_NECP2035.csv` and `midterm4_diag_NECP2035.csv` (the EMPIRE future fleet).

| input (2024 → scenario) | content |
|---|---|
| `Data/BellmanValuesOUT_sddp4[_<label>].csv` | Bellman cuts (a_0 = FR slope, a_1 = ES slope, EU folded into b) |
| `Data/Volume_Scen0_OUT_sddp4[_<label>].csv` | hourly FR/ES reservoir trajectory (EMPIRE year 2015) |
| `results/midterm4_diag[_<label>].csv` | simulated prices / flows / dispatch (20 policy runs) |
| `results/midterm4_sddp[_<label>].log` | SDDP training log (falls back to un-suffixed) |

To regenerate (set `[scenario] label` in `config.toml` first — `"2024"` or an EMPIRE label):
```
julia --project=. midterm_sddp4.jl          # train + export cuts/volumes
julia --project=. midterm_sddp4.jl resim    # rewrite the diagnostics CSV
```


In [13]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
assert (ROOT / "config.toml").exists(), "run this notebook from the repo root"

cfg_text    = (ROOT / "config.toml").read_text(encoding="utf-8")
m4          = cfg_text.split("[midterm4]")[1]
BLOCK_HOURS = int(re.search(r"^block_hours\s*=\s*(\d+)", m4, re.M).group(1))
BGN_DATE    = pd.Timestamp(re.search(r'^bgn_date\s*=\s*"([\d-]+)"', cfg_text, re.M).group(1))
ss          = pd.read_csv("Data/smspp_in/SS_SeasonalStorage.csv", sep=";")
VMAX_ES     = float(ss.loc[ss.Zone == "ES", "MaxVolume"].iloc[0])            # MWh
NB          = 168 // BLOCK_HOURS

# ── scenario selector ────────────────────────────────────────────────────────
# LABEL_OVERRIDE pins which scenario to plot regardless of config.toml.  Set it
# to a label (e.g. "GoRES") to always read that scenario's "_<label>" outputs —
# use this while config.toml [scenario] is being cycled through other scenario
# runs, so the notebook can't silently plot the wrong system.  Leave it None to
# auto-detect from config.toml [scenario] label (whatever midterm_sddp4.jl last
# exported): "2024" reads the un-suffixed files, any other label the suffixed.
LABEL_OVERRIDE = "Trinity"
_m_lab = re.search(r'\[scenario\][\s\S]*?^\s*label\s*=\s*"([^"]+)"', cfg_text, re.M)
LABEL  = LABEL_OVERRIDE or (_m_lab.group(1) if _m_lab else "NECP2035")
SUFFIX = "" if LABEL == "2024" else f"_{LABEL}"
def sfx(p):                                                  # insert _<label> before .ext
    p = Path(p)
    return str(p.with_name(p.stem + SUFFIX + p.suffix))

# palette (validated defaults from the dataviz reference palette)
PAL   = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURF = "#e1e0d9", "#c3c2b7", "#fcfcfb"
SEQ    = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
SEQSC  = [[i / (len(SEQ) - 1), c] for i, c in enumerate(SEQ)]
DIVSC  = [[0.0, "#e34948"], [0.5, "#f0efec"], [1.0, "#2a78d6"]]   # red = import, blue = export
ZONE_C = {"ES": PAL[0], "EU": PAL[1], "PT": PAL[2], "FR": PAL[4]} # fixed slots, never cycled
ZONES  = ["ES", "FR", "PT", "EU"]

def style(fig, title, subtitle=None, h=420, legend_y=1.02, top=70):
    fig.update_layout(
        title=dict(text=title + (f"<br><sup style='color:{INK2}'>{subtitle}</sup>" if subtitle else ""),
                   font=dict(size=16, color=INK), x=0.01, xanchor="left"),
        template="none", paper_bgcolor=SURF, plot_bgcolor=SURF, height=h,
        font=dict(family='system-ui, "Segoe UI", sans-serif', size=12, color=INK2),
        margin=dict(l=70, r=30, t=top, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, xanchor="right", x=1.0),
        hovermode="x unified")
    fig.update_xaxes(gridcolor=GRID, linecolor=AXIS, zeroline=False, ticks="outside", tickcolor=AXIS)
    fig.update_yaxes(gridcolor=GRID, linecolor=AXIS, zeroline=False, ticks="outside", tickcolor=AXIS)
    return fig

def rgba(hexc, a):
    return "rgba(%d,%d,%d,%.2f)" % (*(int(hexc[i:i+2], 16) for i in (1, 3, 5)), a)

# ── load everything (scenario-suffixed) ──────────────────────────────────────
cuts = pd.read_csv(sfx("Data/BellmanValuesOUT_sddp4.csv"))
vol  = pd.read_csv(sfx("Data/Volume_Scen0_OUT_sddp4.csv"))
_diag_path = Path(sfx("results/midterm4_diag.csv"))
assert _diag_path.exists(), (
    f"{_diag_path} not found — generate it with:\n"
    f'    (set config.toml [scenario] label = "{LABEL}")  then\n'
    f"    julia --project=. midterm_sddp4.jl resim")
diag = pd.read_csv(_diag_path)

for z in ZONES:
    diag[f"p{z}"] = diag[f"p{z}"] / BLOCK_HOURS          # per-block duals -> EUR/MWh
diag["date"] = BGN_DATE + pd.to_timedelta((diag.t - 1) * 7 + (diag.b - 1) * BLOCK_HOURS / 24, unit="D")
diag["impES"] = diag.fPTES - diag.fESFR                  # net import into ES [MW]

def week_date(s):                                        # end of cut Timestep s
    if np.isscalar(s):
        return BGN_DATE + pd.Timedelta(days=7 * int(s))
    return BGN_DATE + pd.to_timedelta(np.asarray(s, dtype=int) * 7, unit="D")

V_ES, V_FR = "Hydro|Reservoir_ES_0", "Hydro|Reservoir_FR_0"
vol["date"] = BGN_DATE + pd.to_timedelta(vol.Timestep, unit="h")

# NTC per corridor: the model caps flow at +-fmax, so the observed extreme
# recovers each scenario's rating without hard-coding the 2024 values.
NTC = {c: float(round(diag[c].abs().max())) for c in ("fESFR", "fPTES", "fEUFR")}
print(f"scenario '{LABEL}'  |  {len(cuts)} cuts, {diag.rep.nunique()} simulated runs, "
      f"{diag.t.max()} weeks x {NB} blocks of {BLOCK_HOURS} h")
print(f"NTC [MW]: {NTC}")


AssertionError: results\midterm4_diag_Trinity.csv not found — generate it with:
    (set config.toml [scenario] label = "Trinity")  then
    julia --project=. midterm_sddp4.jl resim

## Training convergence

The lower bound is the SDDP outer approximation of the expected 78-week system cost;
forward-pass simulated costs scatter around the true policy cost.  Converged when the
bound flattens inside the cloud.


In [ ]:
_log = Path(sfx("results/midterm4_sddp.log"))
if not _log.exists():                                    # log naming is manual; fall back
    _log = Path("results/midterm4_sddp.log")
rows = []
for line in _log.read_text().splitlines():
    m = re.match(r"\s*(\d+)[A-Z]?\s+([-\d.e+]+)\s+([-\d.e+]+)\s+([-\d.e+]+)\s+(\d+)", line)
    if m:
        rows.append((int(m.group(1)), float(m.group(2)), float(m.group(3))))
conv = pd.DataFrame(rows, columns=["iteration", "simulation", "bound"])

fig = go.Figure()
fig.add_scatter(x=conv.iteration, y=conv.simulation / 1e9, mode="markers", name="simulated cost",
                marker=dict(size=6, color=MUTED, opacity=0.55))
fig.add_scatter(x=conv.iteration, y=conv.bound / 1e9, mode="lines", name="lower bound",
                line=dict(color=PAL[0], width=2))
fig.add_annotation(x=conv.iteration.iloc[-1], y=conv.bound.iloc[-1] / 1e9,
                   text=f"{conv.bound.iloc[-1]/1e9:.1f} bn", showarrow=False,
                   xanchor="left", xshift=6, font=dict(color=PAL[0]))
style(fig, "SDDP training convergence", "expected 78-week 4-zone system cost [bn EUR]")
fig.update_xaxes(title_text="iteration"); fig.update_yaxes(title_text="bn EUR")
fig.show()


## Value function and water values

The cuts give the expected cost-to-go over the three reservoir states.  The **water
value** is the negative slope of the binding cut: `a_1` for the Spanish reservoir,
`a_0` for the French one (the EU term is already folded into `b`).


In [ ]:
# ES water value surface over (week, V_ES), FR/EU terms at their trajectory values
stages = np.arange(0, int(cuts.Timestep.max()))
vgrid  = np.linspace(0, VMAX_ES, 200)
wv     = np.full((len(stages), len(vgrid)), np.nan)
for i, s in enumerate(stages):
    g = cuts[cuts.Timestep == s]
    val = g.b.values[:, None] + g.a_1.values[:, None] * vgrid[None, :]
    wv[i] = -g.a_1.values[np.argmax(val, axis=0)]

traj = vol.iloc[::168]
fig = go.Figure()
fig.add_heatmap(x=week_date(stages), y=vgrid / 1e6, z=np.clip(wv.T, 0, 120),
                colorscale=SEQSC, colorbar=dict(title="EUR/MWh", outlinewidth=0),
                hovertemplate="week of %{x|%d %b %Y}<br>V_ES %{y:.1f} TWh<br>water value %{z:.1f} EUR/MWh<extra></extra>")
fig.add_scatter(x=traj.date, y=traj[V_ES] / 1e6, mode="lines", name="simulated trajectory",
                line=dict(color=INK, width=2))
style(fig, "ES water value across the horizon",
      "marginal value of stored energy [EUR/MWh] - black line: simulated volume (EMPIRE year 2015)", h=480)
fig.update_yaxes(title_text="ES reservoir volume [TWh]")
fig.update_layout(hovermode="closest")
fig.show()


In [ ]:
# binding water values along each reservoir's own simulated trajectory
def wv_along(slope_col, vol_col):
    out = []
    for s in stages:
        g = cuts[cuts.Timestep == s]
        v = vol.loc[vol.Timestep == min(int(s) * 168, int(vol.Timestep.max())), vol_col].iloc[0]
        out.append(-g[slope_col].values[np.argmax(g.b.values + g[slope_col].values * v)])
    return np.array(out)

fig = go.Figure()
fig.add_scatter(x=week_date(stages), y=wv_along("a_1", V_ES), mode="lines",
                name="ES reservoir", line=dict(color=ZONE_C["ES"], width=2))
fig.add_scatter(x=week_date(stages), y=wv_along("a_0", V_FR), mode="lines",
                name="FR reservoir", line=dict(color=ZONE_C["FR"], width=2))
style(fig, "Binding water value along the simulated trajectory",
      "negative slope of the binding cut at each week's simulated volume [EUR/MWh]")
fig.update_yaxes(title_text="EUR/MWh", rangemode="tozero")
fig.show()


## Reservoir trajectories

Hourly volumes for the deterministic EMPIRE-2015 simulation.  The EU reservoir is a
state of the model but is not part of the exported file format (its value-function
contribution is folded into the cut intercepts).


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Spain", "France"), horizontal_spacing=0.08)
for col, key, z in ((1, V_ES, "ES"), (2, V_FR, "FR")):
    fig.add_scatter(x=vol.date, y=vol[key] / 1e6, mode="lines", name=z,
                    line=dict(color=ZONE_C[z], width=1.5), showlegend=False, row=1, col=col)
style(fig, "Seasonal reservoir volume", "hourly, EMPIRE year 2015 [TWh]", legend_y=1.12, top=95)
fig.update_yaxes(title_text="TWh", col=1)
fig.show()


## Zonal marginal prices

Balance-constraint duals of the simulated policy (20 Monte-Carlo runs over the five
weather years).  Left: weekly means, band = range across runs.  Right: duration curves.


In [ ]:
rep_wk = diag.groupby(["rep", "t"]).agg(date=("date", "first"),
                                        **{f"p{z}": (f"p{z}", "mean") for z in ZONES}).reset_index()
wk = rep_wk.groupby("t").agg(date=("date", "first"), **{
    f"p{z}_{s}": (f"p{z}", s) for z in ZONES for s in ("mean", "min", "max")}).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=("weekly mean, band = range over runs",
                                                    "duration curve"), horizontal_spacing=0.08)
q = np.linspace(0, 100, 400)
for z in ZONES:
    fig.add_scatter(x=wk.date, y=wk[f"p{z}_max"], mode="lines", line=dict(width=0),
                    showlegend=False, hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=wk.date, y=wk[f"p{z}_min"], mode="lines", line=dict(width=0), fill="tonexty",
                    fillcolor=rgba(ZONE_C[z], 0.15), showlegend=False, hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=wk.date, y=wk[f"p{z}_mean"], mode="lines", name=z,
                    line=dict(color=ZONE_C[z], width=2), row=1, col=1)
    fig.add_scatter(x=q, y=np.percentile(diag[f"p{z}"], 100 - q), mode="lines", name=z,
                    line=dict(color=ZONE_C[z], width=2), showlegend=False, row=1, col=2)
style(fig, "Zonal marginal price", "simulated policy, 20 runs [EUR/MWh]", legend_y=1.12, top=95)
fig.update_yaxes(title_text="EUR/MWh", rangemode="tozero", row=1, col=1)
fig.update_yaxes(range=[0, 200], row=1, col=2)
fig.update_xaxes(title_text="share of blocks [%]", row=1, col=2)
fig.show()


## Spanish dispatch

Weekly mean supply serving Spanish demand, averaged over the 20 runs.  Thermal is
bucketed by marginal cost: base ≤ 50 (nuclear, waste, geo), mid 50–100 (coal, lignite,
CCGT, biomass), peak > 100 EUR/MWh (OCGT, oil).  Pumped storage and imports are net
contributions (only their positive part stacks).


In [ ]:
stack = [("Base (nuclear/bio)", "baseES", PAL[4]), ("Renewables", "resES", PAL[3]),
         ("Reservoir hydro", "turbES", PAL[0]), ("Mid (coal)", "midES", PAL[2]),
         ("Peak (gas/oil)", "peakES", PAL[7]), ("Pumped (net)", "stsES", PAL[1]),
         ("Net imports", "impES", PAL[6])]
wd = diag.groupby("t").agg(date=("date", "first"), dem=("dES", "mean"),
                           **{c: (c, "mean") for _, c, _ in stack}).reset_index()
for _, c, _ in stack[-2:]:
    wd[c] = wd[c].clip(lower=0)

fig = go.Figure()
for name, c, color in stack:
    fig.add_scatter(x=wd.date, y=wd[c] / 1e3, mode="lines", name=name, stackgroup="es",
                    line=dict(width=0.5, color=SURF), fillcolor=color)
fig.add_scatter(x=wd.date, y=wd.dem / 1e3, mode="lines", name="demand",
                line=dict(color=INK, width=2))
style(fig, "Spanish supply mix", "weekly mean [GW], average of 20 simulated runs", h=500)
fig.update_layout(legend=dict(orientation="h", y=-0.12, yanchor="top", x=0, xanchor="left"))
fig.update_yaxes(title_text="GW")
fig.show()


## Interconnector exchanges

Weekly mean net flow (band = range over runs) on the three corridors; positive = flow
in the corridor's reference direction.  Bars: share of blocks at either capacity limit.


In [ ]:
links = [("fESFR", "ES -> FR", "ES"), ("fPTES", "PT -> ES", "PT"), ("fEUFR", "EU -> FR", "EU")]
fig = make_subplots(rows=2, cols=3, shared_xaxes=True, row_heights=[0.72, 0.28],
                    vertical_spacing=0.07, horizontal_spacing=0.06,
                    subplot_titles=[l[1] for l in links] + ["% of blocks at a limit", "", ""])
for col, (c, name, z) in enumerate(links, start=1):
    rw = diag.groupby(["rep", "t"]).agg(date=("date", "first"), v=(c, "mean")).reset_index()
    g = rw.groupby("t").agg(date=("date", "first"), m=("v", "mean"),
                            lo=("v", "min"), hi=("v", "max")).reset_index()
    cong = diag.assign(x=(diag[c].abs() >= NTC[c] * 0.999)).groupby("t")["x"].mean()
    fig.add_scatter(x=g.date, y=g.hi / 1e3, mode="lines", line=dict(width=0),
                    showlegend=False, hoverinfo="skip", row=1, col=col)
    fig.add_scatter(x=g.date, y=g.lo / 1e3, mode="lines", line=dict(width=0), fill="tonexty",
                    fillcolor=rgba(ZONE_C[z], 0.18), showlegend=False, hoverinfo="skip",
                    row=1, col=col)
    fig.add_scatter(x=g.date, y=g.m / 1e3, mode="lines",
                    line=dict(color=ZONE_C[z], width=2), showlegend=False, row=1, col=col)
    for s in (1, -1):
        fig.add_hline(y=s * NTC[c] / 1e3, line=dict(color=AXIS, width=1, dash="dot"),
                      row=1, col=col)
    fig.add_bar(x=g.date, y=100 * cong.values, marker_color=ZONE_C[z],
                showlegend=False, row=2, col=col)
style(fig, "Cross-border exchange", "weekly mean, band = range over runs [GW]", h=520, top=95)
fig.update_yaxes(title_text="GW", row=1, col=1)
fig.update_yaxes(title_text="% at limit", range=[0, 100], row=2, col=1)
for col in (2, 3):
    fig.update_yaxes(range=[0, 100], row=2, col=col)
fig.update_layout(bargap=0)
fig.show()


## Maps

The four zones on the European map (the EU zone is drawn over the member countries it
aggregates, for orientation only).  Lines mark the three interconnectors, labelled with
the mean net transfer and the share of blocks at a capacity limit.


In [ ]:
EU_ISO = ["DEU", "BEL", "NLD", "LUX", "ITA", "AUT", "CHE", "DNK", "POL", "CZE",
          "SVK", "HUN", "SVN", "HRV", "ROU", "BGR", "GRC", "SWE", "FIN", "NOR",
          "EST", "LVA", "LTU", "IRL"]
CENTROID = {"ES": (40.2, -3.7), "PT": (39.6, -8.1), "FR": (46.8, 2.4), "EU": (51.2, 13.5)}

def zone_map(zvals, colorscale, cbar_title, title, subtitle, zmid=None, zrange=None):
    locs = ["ESP", "PRT", "FRA"] + EU_ISO
    z    = [zvals["ES"], zvals["PT"], zvals["FR"]] + [zvals["EU"]] * len(EU_ISO)
    fig = go.Figure(go.Choropleth(
        locations=locs, z=z, colorscale=colorscale, zmid=zmid,
        marker_line_color=SURF, marker_line_width=0.6,
        colorbar=dict(title=cbar_title, outlinewidth=0, len=0.7),
        hovertemplate="%{location}: %{z:.1f}<extra></extra>"))
    zlo, zhi = zrange if zrange else (min(zvals.values()), max(zvals.values()))
    for zn, (la, lo) in CENTROID.items():
        rel = 0.5 if zhi == zlo else (zvals[zn] - zlo) / (zhi - zlo)
        dark = rel > 0.6 and colorscale is SEQSC
        fig.add_scattergeo(lat=[la], lon=[lo], mode="text",
                           text=f"<b>{zn}</b><br>{zvals[zn]:.1f}",
                           textfont=dict(color="#ffffff" if dark else INK, size=13),
                           showlegend=False, hoverinfo="skip")
    fig.update_geos(scope="europe", bgcolor=SURF, showcountries=False, showframe=False,
                    landcolor="#f0efec", lataxis_range=[34, 62], lonaxis_range=[-12, 25])
    fig.update_layout(title=dict(text=f"{title}<br><sup style='color:{INK2}'>{subtitle}</sup>",
                                 font=dict(size=16, color=INK)),
                      paper_bgcolor=SURF, height=560, margin=dict(l=10, r=10, t=70, b=10),
                      font=dict(family='system-ui, "Segoe UI", sans-serif', color=INK2))
    return fig

mean_p = {z: diag[f"p{z}"].mean() for z in ZONES}
fig = zone_map(mean_p, SEQSC, "EUR/MWh", "Mean zonal marginal price",
               "simulated policy, all blocks and runs [EUR/MWh]")

arrows = [("fESFR", "ES", "FR", (44.6, -8.2), "middle center"),
          ("fPTES", "PT", "ES", (35.2, -9.8), "middle right"),
          ("fEUFR", "EU", "FR", (53.4, 5.0),  "middle center")]
for c, a, b, (lab_lat, lab_lon), pos in arrows:
    m    = diag[c].mean()                      # >0 : a -> b
    cong = (diag[c].abs() >= NTC[c] * 0.999).mean()
    src, dst = (a, b) if m > 0 else (b, a)
    fig.add_scattergeo(lat=[CENTROID[src][0], CENTROID[dst][0]],
                       lon=[CENTROID[src][1], CENTROID[dst][1]], mode="lines",
                       line=dict(color=INK2, width=1 + 3 * abs(m) / 5000),
                       showlegend=False, hoverinfo="skip")
    fig.add_scattergeo(lat=[lab_lat], lon=[lab_lon], mode="text", textposition=pos,
                       text=f"{src} -> {dst}: {abs(m)/1e3:.1f} GW<br>at limit {cong:.0%} of blocks",
                       textfont=dict(size=11, color=INK), showlegend=False, hoverinfo="skip")
fig.show()


In [ ]:
# seasonal contrast on a shared scale
month = diag.date.dt.month
for name, mask in (("summer (Jun-Sep)", month.isin([6, 7, 8, 9])),
                   ("winter (Dec-Feb)", month.isin([12, 1, 2]))):
    d = diag[mask]
    f = zone_map({z: d[f"p{z}"].mean() for z in ZONES}, SEQSC, "EUR/MWh",
                 f"Mean zonal price - {name}", "simulated policy [EUR/MWh]", zrange=(20, 110))
    f.data[0].update(zmin=20, zmax=110)
    f.show()


In [ ]:
# net position: blue = exporter, red = importer
net = {"ES": (diag.fESFR.mean() - diag.fPTES.mean()) / 1e3,
       "PT": diag.fPTES.mean() / 1e3,
       "FR": -(diag.fESFR.mean() + diag.fEUFR.mean()) / 1e3,
       "EU": diag.fEUFR.mean() / 1e3}
fig = zone_map(net, DIVSC, "GW", "Mean net position",
               "average net export (+) / import (-) [GW], simulated policy", zmid=0.0)
fig.show()


## Summary

In [ ]:
print("mean / median zonal price [EUR/MWh]:")
display(pd.DataFrame({z: {"mean": diag[f"p{z}"].mean(), "median": diag[f"p{z}"].median(),
                          "p5": diag[f"p{z}"].quantile(0.05), "p95": diag[f"p{z}"].quantile(0.95)}
                      for z in ZONES}).round(1))

diag["mon"] = pd.DatetimeIndex(diag.date).to_period("M")
display(diag.groupby("mon")[[f"p{z}" for z in ZONES]].mean().round(1))

E = 1e-6 * BLOCK_HOURS / diag.rep.nunique()
es = pd.Series({"demand": diag.dES.sum(), "renewables": diag.resES.sum(),
                "base thermal": diag.baseES.sum(), "mid thermal": diag.midES.sum(),
                "peak thermal": diag.peakES.sum(), "reservoir hydro": diag.turbES.sum(),
                "pumped (net)": diag.stsES.sum(), "net imports": diag.impES.sum()}) * E
print("78-week Spanish energy balance [TWh], mean over runs:")
display(es.round(1).to_frame("TWh"))


mean / median zonal price [EUR/MWh]:


,ES,FR,PT,EU
mean,17.3,17.6,17.3,17.0
median,18.4,17.9,18.4,17.9
p5,0.0,13.6,0.0,-0.0
p95,24.3,24.3,24.3,24.3


,pES,pFR,pPT,pEU
mon,,,,
2024-07,16.2,15.9,16.2,14.8
2024-08,15.0,15.4,15.0,14.2
2024-09,17.6,16.9,17.6,16.2
2024-10,18.7,18.2,18.7,17.9
2024-11,20.5,20.5,20.5,20.7
2024-12,21.5,20.4,21.5,19.9
2025-01,21.4,21.9,21.4,21.6
2025-02,19.3,20.1,19.3,19.9
2025-03,16.1,17.9,16.1,17.7


78-week Spanish energy balance [TWh], mean over runs:


,TWh
demand,552.2
renewables,417.9
base thermal,95.1
mid thermal,0.0
peak thermal,0.0
reservoir hydro,40.9
pumped (net),-13.1
net imports,4.9
